# 08 - Vision Transformer

Este notebook implementa a tarefa 19 da ordem recomendada: treinar um modelo baseado em Transformer para imagens e comparar com as CNNs.

Modelo principal recomendado: `ViT-B/16` com pesos pre-treinados do torchvision.

Opcao alternativa disponivel no codigo: `Swin Transformer Tiny`.

## Pre-requisitos

Execute antes:

1. `00_download_dataset_kaggle.ipynb`
2. `01_eda_dataset.ipynb`
3. `02_preprocessamento_splits.ipynb`
4. `03_validacao_preprocessamento_dataloaders.ipynb`
5. `04_validacao_pipeline_treino_avaliacao.ipynb`
6. `05_treinamento_cnn_propria.ipynb`
7. `06_treinamento_cnn_padrao.ipynb`
8. `07_treinamento_transfer_learning.ipynb`

Observacao: `pretrained=True` pode baixar pesos do torchvision se eles ainda nao estiverem em cache.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from torch import nn

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataloaders
from src.models.vision_transformer import (
    create_vision_transformer,
    get_trainable_parameter_names,
    transformer_model_summary,
    unfreeze_transformer_final_blocks,
)
from src.training.evaluate import evaluate_model
from src.training.train import TrainConfig, train_model

config.ensure_project_directories()
config.seed_everything()

print("DEVICE:", config.DEVICE)
print("SPLITS_DIR:", config.SPLITS_DIR)

## Configuracao do Experimento

Para o projeto final, rode `vit_b_16`. Se quiser testar uma arquitetura alternativa, troque `MODEL_NAME` para `swin_t`.

In [ ]:
MODEL_NAME = "vit_b_16"  # opcoes: "vit_b_16" ou "swin_t"
PRETRAINED = True
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5

required_splits = [config.SPLITS_DIR / f"{split}.csv" for split in ["train", "val", "test"]]
missing_splits = [path for path in required_splits if not path.exists()]
if missing_splits:
    raise FileNotFoundError(
        "Splits ausentes: "
        + ", ".join(str(path) for path in missing_splits)
        + ". Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

loader_config = DataLoaderConfig(
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    use_weighted_sampler=False,
)
loaders = create_dataloaders(dataloader_config=loader_config)

for split_name, loader in loaders.items():
    print(split_name, "imagens:", len(loader.dataset), "classes:", loader.dataset.class_counts)

## Criacao e teste de forward pass

A cabeca final e substituida por uma saida binaria. A saida esperada e um logit por imagem.

In [ ]:
model = create_vision_transformer(
    model_name=MODEL_NAME,
    pretrained=PRETRAINED,
    freeze_backbone=True,
)

print("Resumo inicial:", transformer_model_summary(model))
print("Parametros treinaveis:", len(get_trainable_parameter_names(model)))

dummy_batch = torch.randn(2, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)
with torch.no_grad():
    dummy_logits = model(dummy_batch)

print("Forward output shape:", tuple(dummy_logits.shape))

## Fase 1: treino da cabeca classificadora

Nesta fase, apenas a cabeca final e treinavel. O backbone Transformer fica congelado.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
head_optimizer = torch.optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=HEAD_LR,
    weight_decay=config.WEIGHT_DECAY,
)

head_train_config = TrainConfig(
    model_name=f"{MODEL_NAME}_head",
    epochs=HEAD_EPOCHS,
    device=config.DEVICE,
    metric_to_maximize="f1",
    use_amp=torch.cuda.is_available(),
)

head_result = train_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=head_optimizer,
    train_config=head_train_config,
)

head_metrics = evaluate_model(
    model=model,
    dataloader=loaders["test"],
    model_name=f"{MODEL_NAME}_head",
    device=config.DEVICE,
    output_dir=config.METRICS_DIR,
    checkpoint_path=head_result["best_checkpoint_path"],
)
head_metrics["phase"] = "head"
head_metrics

## Fase 2: fine-tuning parcial

Liberamos os blocos finais do Transformer para adaptar melhor as representacoes ao dominio de raio-X.

In [ ]:
unfreeze_transformer_final_blocks(model, MODEL_NAME)
print("Resumo fine-tuning:", transformer_model_summary(model))
print("Parametros treinaveis:", len(get_trainable_parameter_names(model)))

fine_tune_optimizer = torch.optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=FINE_TUNE_LR,
    weight_decay=config.WEIGHT_DECAY,
)

fine_tune_config = TrainConfig(
    model_name=f"{MODEL_NAME}_finetune",
    epochs=FINE_TUNE_EPOCHS,
    device=config.DEVICE,
    metric_to_maximize="f1",
    use_amp=torch.cuda.is_available(),
    gradient_clip_norm=1.0,
)

fine_tune_result = train_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=fine_tune_optimizer,
    train_config=fine_tune_config,
)

fine_tune_metrics = evaluate_model(
    model=model,
    dataloader=loaders["test"],
    model_name=f"{MODEL_NAME}_finetune",
    device=config.DEVICE,
    output_dir=config.METRICS_DIR,
    checkpoint_path=fine_tune_result["best_checkpoint_path"],
)
fine_tune_metrics["phase"] = "finetune"
fine_tune_metrics

## Comparacao inicial com CNNs

Se os notebooks anteriores ja foram executados, esta celula carrega metricas existentes em `reports/metricas/` e compara com o Transformer.

In [ ]:
transformer_metrics_df = pd.DataFrame([head_metrics, fine_tune_metrics])
transformer_metrics_path = config.METRICS_DIR / "vision_transformer_metrics_summary.csv"
transformer_metrics_df.to_csv(transformer_metrics_path, index=False)

metric_files = sorted(config.METRICS_DIR.glob("*_metrics.csv"))
all_metric_frames = []
for metric_file in metric_files:
    frame = pd.read_csv(metric_file)
    frame["source_file"] = metric_file.name
    all_metric_frames.append(frame)

if all_metric_frames:
    comparison_df = pd.concat(all_metric_frames, ignore_index=True)
    comparison_path = config.METRICS_DIR / "vision_transformer_vs_existing_models.csv"
    comparison_df.to_csv(comparison_path, index=False)
    display(
        comparison_df.sort_values(["f1", "auc_roc"], ascending=False)[
            [
                "model_name",
                "accuracy",
                "precision",
                "recall",
                "f1",
                "auc_roc",
                "seconds_per_image",
                "total_parameters",
                "trainable_parameters",
            ]
        ]
    )
    print("Comparacao salva em:", comparison_path)
else:
    print("Nenhum arquivo *_metrics.csv encontrado ainda em reports/metricas/.")

print("Metricas do Transformer salvas em:", transformer_metrics_path)

## Proxima etapa

Depois do Transformer, seguir para consolidar metricas, comparar desempenho e eficiencia, e escolher o modelo final.